# Statistical Analysis and Visualization of Maize Production in Zambia Using Python
**ICD 341 Advanced Statistical Computing: Capstone Project**

> **Data note:** `data/maize_production.csv` is *simulated* (see `src/make_data.py`) so the project runs end to end. Replace it with official data (same columns) and re-run all cells to analyse real figures.

## 1. Introduction and objectives
Maize is Zambia's staple crop, so understanding what drives yield and how production responds to weather is important for planning and food security.

**Research question:** How do rainfall, fertiliser use and time affect maize yield across districts, and how risky is production under drier conditions?

**Objectives**
1. Load, clean and organise the dataset.
2. Calculate descriptive statistics.
3. Analyse production patterns across districts and years.
4. Fit appropriate statistical models (OLS regression, fixed effects).
5. Quantify uncertainty (bootstrap) and risk (Monte Carlo simulation).
6. Check for hidden groups in yields with a Gaussian Mixture Model.
7. Visualise results and provide an interactive dashboard (`dashboard/app.py`).
8. Present findings and recommendations.

## 2. Setup

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.append(str(Path.cwd().parent / "src"))   # notebook lives in notebooks/
import analysis as an

np.random.seed(123)            # reproducibility
pd.set_option("display.precision", 3)

## 3. Data loading and cleaning

In [ ]:
raw = an.load_data()
print("Shape:", raw.shape)
print("\nMissing values per column:")
print(raw.isna().sum())
raw.head()

In [ ]:
df = an.clean_data(raw)     # missing rainfall/fertiliser -> district median
print("After cleaning:", df.shape, "| missing values:", int(df.isna().sum().sum()))

**Cleaning decision:** a handful of missing rainfall and fertiliser values were filled with the *district median*. This is simple and keeps all rows, but it understates variability slightly. With real data, document why values are missing before imputing.

## 4. Descriptive statistics

In [ ]:
an.descriptive_stats(df)

In [ ]:
an.district_summary(df)

`cv_yield_pct` is the coefficient of variation of yield: a higher value means less stable yields from year to year.

## 5. Visual exploration

In [ ]:
an.make_figures(df)      # saves PNGs to outputs/
from IPython.display import Image, display
for f in ["production_by_district", "yield_trend", "yield_drivers"]:
    display(Image(filename=f"../outputs/{f}.png"))

## 6. Trend over time

In [ ]:
yearly, res = an.national_trend(df)
print(f"Slope = {res.slope:.4f} t/ha per year, r = {res.rvalue:.3f}, p = {res.pvalue:.2g}")

## 7. Regression models

In [ ]:
table, info, resid = an.ols(df)
print(info)
table

In [ ]:
tfe, ife, resid_fe = an.ols_fixed_effects(df)
print(ife)
tfe

**Interpretation guide**
- Each coefficient is the change in yield (t/ha) for a one-unit increase in that variable, holding the others fixed.
- Adding district dummies (fixed effects) absorbs permanent differences between districts (soil, climate, farming systems), so R² rises and the estimates for rainfall and fertiliser are less contaminated by those differences.
- These are **associations**, not proven causal effects.

In [ ]:
from scipy import stats
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
stats.probplot(resid_fe, plot=ax[0]); ax[0].set_title("Q-Q plot of residuals")
ax[1].scatter(range(len(resid_fe)), resid_fe, s=10); ax[1].axhline(0, color="r")
ax[1].set(title="Residuals by observation", xlabel="Observation")
plt.tight_layout(); plt.show()

Check: points near the Q-Q line suggest roughly normal errors; no obvious pattern in the residual plot suggests no major misspecification.

## 8. Uncertainty: bootstrap confidence interval

In [ ]:
ci, boots = an.bootstrap_ci(df["yield_t_ha"], B=5000)
print(f"Mean yield = {df['yield_t_ha'].mean():.3f} t/ha, 95% bootstrap CI [{ci[0]:.3f}, {ci[1]:.3f}]")
plt.hist(boots, bins=40, edgecolor="black"); plt.axvline(ci[0], color="r"); plt.axvline(ci[1], color="r")
plt.title("Bootstrap distribution of mean yield"); plt.xlabel("t/ha"); plt.show()

## 9. Risk: Monte Carlo simulation under rainfall scenarios

In [ ]:
base = an.monte_carlo_production(df)
dry  = an.monte_carlo_production(df, rain_shift=-100)
print(f"Expected production: baseline {base.mean():,.0f} t, drier {dry.mean():,.0f} t "
      f"({100*(dry.mean()/base.mean()-1):+.1f}%)")
print("5th percentile baseline / drier:", np.percentile(base, 5).round(), np.percentile(dry, 5).round())
plt.hist(base, bins=60, alpha=.6, label="Baseline"); plt.hist(dry, bins=60, alpha=.6, label="Rainfall -100 mm")
plt.legend(); plt.xlabel("Tonnes"); plt.title("Simulated production, average district"); plt.show()

The simulation draws rainfall and fertiliser from normal distributions fitted to the data, pushes them through the fitted regression, and adds residual noise. It assumes the model is correct and that conditions stay within the observed range.

## 10. Gaussian Mixture Model for yield groups

In [ ]:
gmm, bics, labels = an.gmm_yield_groups(df)
print("BIC by number of components:", {k: round(v, 1) for k, v in bics.items()})
print("Best number of components (lowest BIC):", gmm.n_components)
print("Means:", gmm.means_.ravel().round(2), "Weights:", gmm.weights_.round(2))

If BIC selects one component, yields form a single group rather than distinct clusters. That is a valid finding, not a failure of the method.

## 11. Conclusions and recommendations
1. Rainfall and fertiliser are positively associated with yield, and yields trend upward over time (see Section 7 for estimates and p-values).
2. Districts differ systematically; fixed effects improve model fit substantially.
3. Drier-season scenarios reduce expected production and raise downside risk (Section 9).
4. Recommendations: support rainfall-risk management (drought-tolerant seed, irrigation, insurance) and improve fertiliser access in low-yield districts.

**Limitations**
- The bundled data is simulated; conclusions about real Zambian districts require real data.
- OLS ignores spatial and temporal correlation; a mixed-effects or panel model would be a next step.
- Imputation by median is a simplification.

**Reproducibility:** fixed seeds, `requirements.txt`, and a fixed folder layout. Re-run: `python src/make_data.py`, then `python src/analysis.py`, or run this notebook top to bottom.

## 12. References
- Cameron, A.C. & Trivedi, P.K. *Microeconometrics*; OLS and fixed-effects methods.
- Efron, B. & Tibshirani, R. (1993). *An Introduction to the Bootstrap*.
- Course notes: ICD 341 Advanced Statistical Computing, Mulungushi University.
- Replace with data source citation (e.g. CSO/MoA Crop Forecast Survey, FAOSTAT) when using real data.